# 多维度调试——三个真实缺陷的复现与修复

> **多维度调试**（建议 ~1.5h，含实操）。产物 ②：三坑的根因、修复、回归三件套。前置要求：03.01 节完成。素材：`src/gelu_approx_naive/`（真机验证 4 败 10 过）。

> **本节新增知识只有 2 件**：① **症状分诊**（四类问题定族）② **根因分层**（算法 / API 约束 / 指令三层定位）。修复手段没有一件是新的。分段，就是把基础课程的容差概念用到多步链上；恒等式替换，就是"换个等价公式再算一遍"。全是已有知识，用在真实失败上。

> **写给第一次接触算子的读者**：算子代码由 AI 团队编写与修复，指挥官角色看得懂结果、做得了决定就够。

<details><summary>📖 第一次见这些词？点开 30 秒速览</summary>

| 词 | 白话 |
|---|---|
| **golden** | 参考答案：用双精度浮点在 CPU 上算出的标准值，测试拿它判卷 |
| **kernel** | 算子核心代码：真正跑在 NPU 上的计算函数 |
| **naive** | 朴素版：故意不做任何优化的直译版实现，用来暴露问题 |
| **容差（rtol/atol）** | 允许的误差范围：算得和参考答案差多少以内算对 |
| **黑盒用例** | 只看输入输出对不对的测试题，不关心内部怎么实现 |

</details>


**本模块目标**：
1. 区分**数学正确**与**浮点正确**，复现代码没错但超差的现象；
2. 掌握**二维定位**：先按症状分诊（判断是哪一类问题），再按由浅入深找根因在哪一层：**算法层 → API 约束层 → 指令层**；
3. 完成分层修复并用回归验证：真机实测每个修复各让哪些用例转绿。

> 素材的完整验证记录见 `src/gelu_approx_naive/FACTS.md`（真机实测，本节回归声明的唯一依据）。

## 1. 正确的两层含义

先看朴素版（naive：故意不做任何优化的直译版实现）`src/gelu_approx_naive/`，真机验证 **4 败 10 过**。用例编号 L0-x/L1-x/L2-x 沿用黑盒测试三级分级（基础课程 02.01 实录的 40/40 即 L0 基础 / L1 常规 / L2 边界），核心计算：

```cpp
cubed = x * x * x;              // x³
cubed *= 0.044715;
inner = x + cubed;
tanhIn = inner * 0.7978845608;  // √(2/π)
tanhOut = tanh(tanhIn);
result = x * (tanhOut + 1);
z = result * 0.5;
```

它**数学正确**：公式与 golden（参考答案：双精度 CPU 算出的标准值）完全一致，编译零警告，10 个用例结果全对。

但测试里的正确是另一回事。**浮点正确**要求每个输入的误差都落在容差内。基础课程 01.02 的判据 rtol=1e-5 / atol=1e-7，现在从"验收时用一次"变成"逐个输入检查"。add_custom 的加法是 IEEE 754 精确运算（验收时 17/17 位级一致，max_diff=0.0），精度从来不是话题；但 tanh 是近似运算，每一步都带舍入误差。03.01 节的实验已经预警：8 步链在负值暴露带的预算只剩 ~23% 裕量，任何额外误差源都会越线。

**数学正确 + 浮点不正确 = 今天的调试对象。** 先在 CPU 上复现这个悖论：运行下面的 cell，看五个输入域里哪些已经超差：


In [ ]:
# "代码没错但超差"——CPU 复现 naive 8 步链的精度问题（无 NPU）
import numpy as np
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')) if os.path.exists('src') else '.', 'src/gelu_approx_naive/tests'))

def gelu_naive_fp32(x):
    """naive 8 步链（无分段、无精度保护）"""
    c = x * x * x
    c = c * np.float32(0.044715)
    inner = x + c
    t = inner * np.float32(0.7978845608)
    th = np.tanh(t)
    r = x * (th + np.float32(1.0))
    return r * np.float32(0.5)

def gelu_golden(x):
    x64 = x.astype(np.float64)
    inner = np.sqrt(2.0/np.pi) * (x64 + 0.044715 * x64**3)
    return (0.5 * x64 * (1.0 + np.tanh(inner))).astype(np.float32)

rng = np.random.default_rng(42)
print("naive 链精度测试（14 个用例域，CPU 模拟）")
print("═" * 55)
print(f"{'用例域':<16s} {'样本数':>6s} {'中位diff':>10s} {'最大diff':>10s} {'超差%':>6s}")
print("─" * 55)
domains = [
    ("常规 [-3,3]", rng.uniform(-3, 3, 5000)),
    ("小值 [0,1e-4]", rng.uniform(0, 1e-4, 5000)),
    ("精度区 [5,20]", np.linspace(5, 20, 5000)),
    ("负值区 [-20,-5]", np.linspace(-20, -5, 5000)),
    ("饱和区 [20,100]", np.linspace(20, 100, 5000)),
]
for name, x in domains:
    x = x.astype(np.float32)
    z = gelu_naive_fp32(x)
    g = gelu_golden(x)
    d = np.abs(z.astype(np.float64) - g.astype(np.float64))
    tol = 1e-7 + 1e-5 * np.abs(g.astype(np.float64))
    pct = np.sum(d > tol) / len(d) * 100
    print(f"{name:<16s} {len(x):>6d} {np.median(d):>10.2e} {np.max(d):>10.2e} {pct:>5.1f}%")
print("─" * 55)
print("→ CPU 上已可见：负值区的精度问题真实存在——「代码没错但超差」")
print("→ 注意中位/最大误差都刚超 atol 一点点：预算越线，不是算错")
print("→ NPU 向量指令的精度比 np.tanh 更低——坑 3 实测对比")

## 2. Lab B：排查路线——先定类，再分层

失败已复现。下一步不是改代码，而是定位根因。定位有两个**正交维度**：先按症状定类（哪一族问题），再按深度分层（哪一层根因）。
<img src="./images/03.02_2d_triage.png" alt="二维定位矩阵：症状四类（定族）× 根因三层（定深度）" width="700px">


### 第一步：症状初判，四类问题分诊

基础课程 01.03 教过按症状找 Skill（路由表雏形），这里把症状维度**补全**为完整的四类分诊，每类给出表现、信号与定位入口：

| 类 | 典型症状 | 定位入口 |
|---|---|---|
| ① 环境 | 算子尚未运行即被阻断：设备识别失败、工具链缺失、CANN 版本/路径异常（信号：npu-smi 无设备、ASCEND_HOME_PATH 未配置） | `ascendc-env-check` |
| ② 运行时错误 | 已调度执行但 ACLNN 返回非零错误码（速查：161xxx=参数错 / 507035=核异常 / 561002=Tiling 错 / 561003=Kernel 未找到） | `ascendc-runtime-debug` |
| ③ 精度 | 可运行完成但结果不对：全零、随机值、误差超阈、NaN、Cast 后错乱（交叉验证：FP32 通过但 FP16/BF16 失败 → 指向半精度转换环节） | `ascendc-precision-debug` |
| ④ 崩溃卡死 | 无法运行完成：卡死挂起、Kernel 超时、SegFault、Abort、偶发崩溃（手段：coredump+gdb 看 bt；卡死查 plog；偶发用 msSanitizer 查内存越界） | `ascendc-crash-debug` |

**通用纪律**（承接基础课程 01.03 的求助纪律）：先固定最小可复现用例 → 查官方文档对照 → 清 `build/` 与 `kernel_cache/` → 快速法 ≤7 次未果即切二分调试。

**今天的 4 个失败**：能跑完（排除 ④）、无错误码（排除 ②）、环境就绪（排除 ①）、特定输入域超差，属**③ 精度问题**。分诊完成，进入第二步。

### 第二步：深度分层，从离代码最近的层次开始

根因可能藏在三个深度（上图横轴），按离所写代码的距离由浅入深：
- **算法层**：所写的公式和策略；
- **API 约束层**：平台的使用约定（对齐 / 静默截断（不报错、悄悄丢数据）/ 语义）；
- **指令层**：硬件指令本身的精度规格。

两条原则：
1. 教学顺序从算法层逐层深入，每层修掉该层的问题，剩下的失败属于更深层；
2. 实战中签名明确时可直接跳查：失败位置在 64 元素边界后 → API 层；diff 刚超 atol 且集中在特定输入范围 → 指令层。

今天三个坑按浅→深排布：**坑 1 算法层（20min）→ 坑 2 API 约束层·实录走读（15min）→ 坑 3 指令层（20min）**。

> **执行位置**：以下命令在课程仓库 `tutorials/CANNBot/03_advanced_path/` 目录下执行（素材工程在 `src/` 子目录）。

### 坑 1（20min）· 算法层：预算紧张与分段策略

**为什么先查这里**：03.01 节实验测得，纯 8 步舍入累积的带内最大误差 **7.7e-8**，距 atol=1e-7 只剩 23% 裕量。预算紧张是结构性的：负值带（x∈[-6,-3]）会发生**抵消放大**。近数相减，小误差占比暴涨：就像 100.0001 − 100.0000，末位的微小偏差在差值里占大头。这个被放大的误差再乘上 |x|≈3~6，传到输出。

<details><summary>📖 选读：抵消放大的数学形态</summary>

`δ = 1+tanh(y) ≈ 1e-7~1e-4` 是两个近似相等数之差，tanh 的绝对误差（~ulp）在 δ 中的占比被相对放大一到四个数量级，随后经 `z = 0.5·x·δ` 传到输出。这是基础课程 rtol/atol 判据看不到的结构：判据管"总误差 vs 容差"，抵消管"误差在链内如何被放大"。

</details>

**分析**：运行下面的逐步误差曲线，看误差在哪几步跳变（x³ 的连乘和 tanh 步之后），验证误差不是均匀增长、而是在特定步骤集中跳变：


In [ ]:
# 坑 1（算法层）分析：8 步链逐步误差增长曲线——误差在哪些步骤跳变
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

steps = ['x²', 'x³', '×0.045', '+x', '×√(2/π)', 'tanh', '+1', '×x', '×0.5']
x_test = np.array([1.234, 5.678, 12.34, 18.75], dtype=np.float32)

fig, axes = plt.subplots(1, len(x_test), figsize=(14, 3), sharey=True)
fig.suptitle('8步链逐步误差增长（|fp32 - fp64|）', fontsize=11, y=1.02)

for ax, xv in zip(axes, x_test):
    x32 = np.float32(xv); x64 = np.float64(xv)
    err32, err64 = [x32], [x64]
    c32 = x32; c64 = x64
    for op in ['sq', 'mul_x', 'mul_c', 'add', 'mul_k', 'tanh', 'add1', 'mul_x2', 'mul_h']:
        if op == 'sq': c32 = x32*x32; c64 = x64*x64
        elif op == 'mul_x': c32 = c32*x32; c64 = c64*x64
        elif op == 'mul_c': c32 = c32*np.float32(0.044715); c64 = c64*0.044715
        elif op == 'add': c32 = x32+c32; c64 = x64+c64
        elif op == 'mul_k': c32 = c32*np.float32(0.7978845608); c64 = c64*np.sqrt(2/np.pi)
        elif op == 'tanh': c32 = np.float32(np.tanh(c32)); c64 = np.tanh(c64)
        elif op == 'add1': c32 = c32+np.float32(1); c64 = c64+1
        elif op == 'mul_x2': c32 = x32*c32; c64 = x64*c64
        elif op == 'mul_h': c32 = c32*np.float32(0.5); c64 = c64*0.5
        err32.append(c32); err64.append(c64)
    errs = [abs(float(a) - b) for a, b in zip(err32, err64)]
    ax.bar(range(len(errs)), errs, color='#C7000B', alpha=0.7)
    ax.set_title(f'x={xv}', fontsize=9)
    ax.set_xticks(range(len(steps)))
    ax.set_xticklabels(steps, rotation=60, fontsize=6)
    ax.tick_params(axis='y', labelsize=7)

axes[0].set_ylabel('|fp32-fp64|', fontsize=8)
plt.tight_layout()
plt.savefig('error_growth.png', dpi=100, bbox_inches='tight')
plt.show()
print("→ 误差在 x³ 和 tanh 步骤跳变最大（与讲义分析一致）")

**修复（分段）**：需求卡的实现红线即为此设：|x| > 6 时跳过整条计算链（正侧 z=x，负侧 z=0·x）。阈值取 6 的依据一句话：截断误差（|z_true(±6)| ≤ 1e-10）远小于 atol，把预算消耗直接归零。

```bash
cd src/gelu_approx_naive
patch -p1 < fixes/fix1_segmentation.diff    # 或对照参考 diff 自己实现
bash build.sh --install
cd tests && python3 -m pytest test_gelu_approx_blackbox.py -q
```

**回归（真机实测，预期现象）**：失败集**不变**，仍是 L0-4 / L1-2 / L1-3 / L2-5。

> **修了还是 4 个失败？** 不是修错了：分段收益在饱和区，从碰巧正确变成构造性正确（从公式上就保证对，不靠运气），而这 4 个失败全在 |x|<6 负值带，分段不触碰。**结论：失败不属于算法层**，进入更深一层。

> **提示**：分段负侧若写字面 `z=0`，L2-2（x=−Inf）会**新增**失败。分段策略必须对照 golden 的特殊值行为设计，不能只看数学（完整案例见选读框）。

<details><summary>📖 选读：特殊值陷阱：为什么负侧必须写 0·x 而不是 0</summary>

golden 参考链在 x=−Inf 处因 `Inf·0=NaN` 给出 **NaN**（而非数学上的 0）。参考 diff 用 `0·x` 构造：有限负值→∓0（保号），−Inf→NaN，与 golden 语义精确一致（真机验证）。这正是数学正确之外还有参考语义正确的又一例，是基础课程 01.02 特殊值断言（±0/±Inf/NaN 逐位比对）在完整版工作流里的延续。

</details>

### 坑 2（10min）· API 约束层：Compare 256B（实录走读）

gelu 的朴素版 kernel（算子核心代码，真正跑在 NPU 上的计算函数）里没有 Compare 调用，这一层的坑不自己踩，改为**走读一段真实实录**：正是**基础课程 02.01 的 xlog1py 3 次重试自愈实录**的调试细节。基础课程里用决策者视角看结果（17/22→22/22 自愈），现在换调试视角看根因。AI 团队实现掩码逻辑用的 Compare+Select 记录了两个平台约束：

1. **256B 静默截断**：Compare 的 count 模式要求 count×4B 是 256B 的整数倍（fp32 即 64 个元素一组）。不满足时不报错，**尾部元素被悄悄丢弃**。不报错的约束，比报错的 bug 更危险。坑 1 参考 diff 里的 `computeCount` 64 对齐正是这条经验的直接应用；
2. **NaN 比较语义**：比较指令遇到 NaN，无论怎么比都返回假，NE(y,y) 这种惯用判断会失效，要改用 EQ 自反比较（让 y 和自己比相等）。

以下实录摘自课程开发时的节点 2 实现记录（xlog1py 完整版工作流，Ascend 910B3 + CANN 9.1.0 真机运行；完整台账未随仓发布）。

**走读任务（5min）**：展开下面的实录摘录，找到 Compare 256B 与 NaN 语义这两段原文，记下 ① 实测依据是什么 ② 若不做 64 对齐，哪些用例会静默失败，构成产物 ② 的坑 2 三件套（约束条款 + 实录引用 + 签名推演）。

<details><summary>📖 实录摘录：节点 2 实现记录（§2.3 Tiling 与对齐约束 · §3 调试记录）</summary>

### 2.3 Tiling 与对齐约束

- 多核：`numBlocks = min(coreNumAiv, ceil(N/1024))`，`blockLength = ceil(N/numBlocks)`；核数/UB 容量运行时查询（B1 红线，无硬编码）。
- tile：按运行时 UB 动态计算（预算 76 B/元素：队列 24 + 12 个 float 缓冲 48 + 3 掩码 3，预留 4KB 余量），上限 2048、向下取 64 倍数、下限 64。
- **Compare/Compares count 模式约束（CANN 9.1.0 头文件 CompareCompute/CompareScalarCompute 实测依据）**：按 `count*sizeof(T)/256` 整除分解 repeat，fp32 下 count 非 64 倍数时尾部元素被静默丢弃——全量 tile 为 64 倍数，尾块计算数向上取整到 64 倍数、搬运仍按有效数（CopyOut 仅写有效元素，对齐部分残留数据无副作用）。
- 掩码缓冲按每元素 1 字节分配（保守上界，兼容位压缩布局，规避布局歧义风险）。

---

## 3. 调试记录（ascendc-precision-debug，含失败与根因）

按 Skill 流程执行：固定最小复现（特殊值 15 点探针，`/tmp/opencode/xlog1py_probe.py`）→ 数值验证法定位 → 修复 → 复测。

| # | 现象 | 根因 | 修复 |
|---|------|------|------|
| 1 | 首次构建链接失败 `undefined symbol: AscendC::Subs<...>(...)(.vector)` | dav-2201 无 SubsImpl（dav_c220 binary_scalar 实现仅有 Adds/Muls/Maxs/Mins/Shift/LeakyRelu，头文件声明与实现不匹配） | `r = u−1` 改为常量 1.0 张量（Duplicate 预填充一次）+ `Sub(r, u, ones)`（日志 2-build.log） |
| 2 | 二次构建语法错误（全角字符进入标识符） | 插入注释块缺少 `/*` 起始符 | 补 `/*`（日志 2-build-rebuild.log） |
| 3 | 黑盒首轮 17/22：L0-4/L0-7/L1-2/L1-3/L1-5 均 stage-1 NaN 位失配（期望 NaN、实际非 NaN） | 探针定位：x=±0∧y=NaN→0.0、x=NaN→0.0、其余数值语义全对 ⇒ **c220 vcmp 含 NaN 比较恒为假**（NE(NaN,NaN)=NE(NaN,0)=false 实测；若 NE=NOT(EQ) 将推出 EQ(NaN,0)=true 的荒谬结论，故排除）；NE 惯用法漏检全部 NaN 位，规则1 的 NE(x,0) 使 x=NaN 误选 +0 | 规则链等价重构为仅用 EQ 比较（§2.2）：`EQ(x,0)`（NaN→假，走 z_raw 传播）+ `EQ(y,y)`（仅 y=NaN→假，实现规则2 覆盖）；新增 zeros 常量张量，两级 Select 均 TENSOR_TENSOR（日志 2-blackbox-run.log） |

修复后：探针 15/15 OK（含 3 个原失配位）、黑盒 **22/22 通过**（2-blackbox-run2.log）。调试尝试 3 次（<7 次阈值），每步以探针数值证据收敛。

**给后续节点的经验**：① c220 比较类 API 含 NaN 操作数恒返回 0，NaN 语义须用 EQ 自反比较（y,y）或算术传播实现，禁用 NE 惯用法；② c220 标量二元运算仅 Adds/Muls/Maxs/Mins/Shift/LeakyRelu（无 Subs/Divs），常量减法需常量张量 + 向量减法；③ Compare/Compares count 模式 fp32 count 须 64 倍数（整除分解静默丢尾）。

</details>

### 坑 3（20min）· 指令层：Tanh 精度，压垮预算的最后一根稻草

> **术语卡（指令层速懂：基础课程 rtol/atol 判据之下，第一次看指令的准头）**：
>
> | 术语 | 一句话 | 生活化类比 |
> |---|---|---|
> | **指令精度** | 一条向量指令的结果与数学真值的偏差 | 两台计算器算同一个 tanh，末位不一样：都对，但准头不同 |
> | **恒等式替换** | 用等价公式换一条更准的指令实现 | tanh(y)=2·sigmoid(2y)−1：换个算法算同一个数，绕开不准的那条 |
> | **抵消放大** | 近数相减，误差占比暴涨 | 100.0001 − 100.0000：末位小误差在差值里占大头（坑 1 已见） |

**失败签名**：4 个失败用例的超差元素**全部**在负值带。L2-5 有 189 个超差元素，x∈[-5.12, -2.95]（100% 负值、散布全张量）；L0-4 / L1-3 的唯一超差元素都是特殊值 x=−3；L1-2 的唯一超差元素 x=−5.12（实现算出 −0，参考答案是 −1.02e-07）。diff 量级 1.0~1.4e-7，**刚超 atol 一点点**。

**根因**是两件事叠加：坑 1 的预算只剩 23%；Tanh 指令自身带 ~1 ulp（约 1.2e-7）的近似误差。经抵消放大，误差越过红线。证据链（真机 + CPU 对照，完整数据见 FACTS.md）：

| tanh 实现 | 负值带超差元素 | 带内最大误差 |
|---|---|---|
| 精确 tanh（纯 8 步累积） | 0 | 7.68e-08 |
| np.tanh(fp32)，CPU | 167 | 1.50e-07 |
| NPU `Tanh` 指令，naive（真机） | **189** | **1.39e-07** |

同一根因、三种实现：tanh 的**任何** fp32 实现都带绝对误差（0.5~1 ulp），只有大小的差别。4.1 的 CPU 复现（np.tanh 负值区超差），正是这根链条的中间一环。

**排查**：把失败样本贴给 Agent（触发 `ascendc-precision-debug`），按 Skill 流程走到查指令精度表一步：dav-2201 的 Tanh 指令精度规格不满足本算子负值带的预算要求。

**修复（恒等式绕开）**：用精度更高的 Sigmoid 指令实现 tanh：`tanh(y) = 2·sigmoid(2y) − 1`。这个写法让 sigmoid 的输出远离饱和区，不再发生抵消，最终只剩一次舍入（0.5 ulp），带内最大误差回到 7.68e-8，小于 atol。

```bash
cd src/gelu_approx_naive
patch -p1 < fixes/fix3_sigmoid.diff    # 或自己实现恒等式替换
bash build.sh --install
cd tests && python3 -m pytest test_gelu_approx_blackbox.py -q
```

**回归（真机实测）**：**14/14 全绿**，单独这一项修复即全量通过。至此三层走完：算法层修复提升了健壮性（但失败不属于它），API 层约束在修复实现中拦路（实录为证），指令层才是这 4 个失败的真凶。

三个坑的推进与回归结果，一张图收尾：

<img src="./images/03.02_pit_flow.png" alt="分层修复推进：症状定位 → 坑1 分段（失败集不变）→ 坑2 Compare 实录走读 → 坑3 sigmoid（14/14 全绿）" width="700px">

## 3. 三坑的层次观

| 坑 | 层次 | 症状签名 | 一句话教训 |
|---|---|---|---|
| 分段与特殊值 | **算法层** | 饱和区依赖溢出/饱和语义碰巧正确；负侧字面 0 → x=−Inf 处 L2-2 新失败 | 预算紧张是结构性前提，分段对照 golden 特殊值语义设计；**分段修复 ≠ 全绿**（失败可能属于更深层） |
| Compare 256B / NaN 语义 | **API 约束层** | 尾块元素**静默**丢失（无报错、用例几乎全对）；NE(y,y) 掩码对 NaN 失效 | 静默截断比报错危险，约束要查头文件/实录取证，边界用例是唯一防线 |
| Tanh 指令精度 | **指令层** | diff 1.0~1.4e-7 · 100% 负值带（x∈[-5.12,-2.95]）· 189 元素散布 | 近似指令不能进高精度路径，恒等式换精度更高的指令 |

**排查心法**：从离代码最近的层次开始（算法 → API → 指令），每层修掉该层问题；失败签名明确时可跳查。三个层次不是孤立的。今天的 4 个失败 = 算法层的紧张预算（坑 1）× 指令层的近似误差（坑 3）；API 层的约束则在修复实现中拦路（坑 2）。**复合根因，分层拆解**，这正是多维度调试的核心。

## 4. 要点回顾

**本章数字锚点**：2 维定位（4 类症状 × 3 层根因）· 4 败 10 过 · 14/14 全绿

- 正确分两层：**数学正确**（公式一致）≠ **浮点正确**（逐输入落容差内）；排查**二维定位**：先分诊四类症状，再分层（算法 → API → 指令），签名明确可跳查；
- 三坑一句话：坑 1 分段对照 golden 特殊值语义（修复≠全绿，失败属更深层）；坑 2 Compare 静默截断实录取证；坑 3 Tanh ~1 ulp × 抵消放大越线，sigmoid 恒等式替代 → **14/14 全绿**；
- 证据链纪律：现象 + 上下文 + 已试过 + 日志路径。

**关联术语**：golden · 容差 · Tanh 精度 · Compare 256B · 精度预算 · 抵消放大

**下一站**：[03.03 · 性能工程](./03.03_performance_engineering.ipynb)。正确性已验收，接下来追问跑得快不快、值不值得优化。


## 5. 带走物

| 类别 | 内容 |
|---|---|
| **一句话** | 先定类再分层，修一层验一层 |
| **工具** | 四类症状分诊表 + 三层根因定位（算法→API→指令） |
| **产物** | 三坑修复三件套（根因 + diff + 回归证据） |
| **模板** | 排查路线：症状→分诊→分层→修复→回归；证据链纪律（现象+上下文+已试过+日志） |
| **下一步** | [03.03](./03.03_performance_engineering.ipynb)：正确了，那快不快 |

## 6. 章末检查点

1. **（单选）** 输出精度差 1e-7 量级且集中在特定输入域，最先查什么？
   - A. 内存泄漏  B. 指令精度表  C. 网络配置  D. 编译器版本

2. **（单选）** Compare 类 API count 不满足 256B 对齐时的行为？
   - A. 抛出错误  B. 静默截断  C. 自动对齐  D. 返回空

3. **（简答）** gelu_approx 的 8 步计算链中，如何定位是哪一步引入了主要误差？

<details><summary>参考答案</summary>

1. **B**（精度类症状先查指令规格） 2. **B**
3. 分段 dump 法：在每步之后将中间值写回 GM，与 float64 对应中间值对比，画出逐步误差增长曲线，跳变最大的步骤即主要误差源。
</details>

## 7. 产物自查表（交付 ②）

| 坑 | 根因一句话 | 修复 diff | 回归证据 |
|---|---|---|---|
| 坑 1 分段与特殊值（算法层） | ☐ | ☐ | ☐（失败集不变 + 饱和区构造性正确） |
| 坑 2 Compare 256B（API 层·实录走读） | ☐ | 实录引用 ☐ | 违约束签名推演 ☐ |
| 坑 3 Tanh 指令精度（指令层） | ☐ | ☐ | ☐（14/14 全绿） |

## 8. 课后练习

> 练习答案：[`answer/03.02_answer.txt`](./answer/03.02_answer.txt)

1. **（单选题）** 失败签名为 diff 1.0~1.4e-7、100% 负值带（x∈[-5.12,-2.95]）、散布全张量，按二维定位应归入？
   - A. ② 运行时错误 × API 约束层
   - B. ③ 精度 × 指令层（Tanh 精度 × 抵消放大）
   - C. ③ 精度 × 算法层（分段缺失）
   - D. ④ 崩溃卡死 × 任意层

2. **（多选题）** 坑 1 分段修复后失败集不变（仍 4 败），正确的理解包括？
   - A. 修复写错了，应该回滚
   - B. 4 个失败在 |x|<6 负值带，分段根本不触碰它们，失败不属于算法层（设计内教学点）
   - C. 分段的收益在饱和区：L1-4/L2-3 从碰巧正确变为构造性正确
   - D. 说明 Compare 256B 才是真凶，应该先修坑 2

3. **（简答）** 分段负侧为什么必须写 `0·x` 而不是字面 `0`？从 golden 参考链在 x=−Inf 处的行为（`Inf·0=NaN`）说明分段策略必须对照 golden 特殊值语义设计。

4. **（编程实践·必做）** **修复顺序倒置实验**：对照 `FACTS.md` 的验证矩阵，按相反顺序复现：先应用 `fix3_sigmoid.diff`（跑测试记录失败集），再叠加 `fix1_segmentation.diff`（再跑）。回答：最终结果与 FACTS 记录是否一致？为什么修复顺序不影响终态、却影响中间观察？

5. **（编程实践·选做）** **一招见效的检验用例设计**：为黑盒测试设计一个新用例 L2-6，它能区分分段正确与分段缺失且负侧字面 0 两种实现（提示：输入域要同时覆盖 −Inf、有限大负值与 [-6,-3] 带）。写出用例数据与预期区分逻辑。
